In [1]:
# Colab compatibility: install packages not preinstalled there (no-op locally / on Colab reruns)
import importlib.util, subprocess, sys
for pkg, mod in [('faiss-cpu', 'faiss'), ('lightgbm', 'lightgbm')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# 04 - Two-Tower Retrieval

**From scratch in PyTorch.** In-batch negatives, softmax cross-entropy, FAISS retrieval - plus the item embeddings that power the webapp's "Similar items" and "Because you liked X" sections.

## Architecture - and one key design decision

Two separate networks map users and items into the **same embedding space**; relevance = dot product. Because the towers never interact until the final dot product, all item embeddings can be **precomputed and indexed in FAISS** - that's what makes two-tower a *retrieval* architecture (vs a cross-encoder, which must score every pair at request time).

**Our user tower consumes the user's history, not a user ID.**

$$u = \text{MLP}_{user}\Big(\text{mean}\big(\{e_{i} : i \in \text{last } 20 \text{ liked items}\}\big)\Big) \qquad
v = \text{MLP}_{item}(e_{i})$$

Why history instead of a user-ID embedding (defense question - know this cold):

1. **The webapp requires it.** The online flow is: fetch history from PostgreSQL → *forward pass through the user tower* → FAISS. With an ID embedding, any user not in the training table is a dead end. With a history tower, anyone with ≥1 interaction gets a real-time, up-to-date embedding - including test-period activity the model never saw.
2. **Cold-start.** New-ish users (few interactions, or joined after training) still get personalized retrieval. MF-BPR fundamentally cannot do this.
3. It's the honest comparison: the instructor's sanity check is "if your two-tower can't beat MF (IDs only), something is wrong." Feeding the tower richer input (history) is exactly the point of the architecture.

Both outputs are **L2-normalized**, so dot product = cosine similarity, and a temperature $\tau$ scales the logits.

## The loss: in-batch softmax cross-entropy

For a batch of $B$ (user, positive item) pairs, we compute all $B \times B$ scores $S = UV^\top/\tau$. Row $k$ is a $B$-way classification: the "correct class" is item $k$, and **the other $B-1$ items in the batch act as negatives** - free, no sampling pass needed:

$$\mathcal{L} = -\frac{1}{B}\sum_k \log \frac{e^{u_k \cdot v_k/\tau}}{\sum_{l} e^{u_k \cdot v_l/\tau}}$$

Two subtleties we handle explicitly (both are classic defense questions):

- **Sampling bias / log-Q correction** *(bonus: sampling-bias correction)*. In-batch negatives are drawn from the batch, i.e. proportionally to item popularity - popular items get punished as negatives far too often. Correction: subtract $\log Q(i)$ (the item's sampling probability) from each logit. Popular items keep competitive scores; the softmax becomes an unbiased estimate of the full softmax.
- **False negatives from duplicate targets.** If item $i$ appears twice as a target in one batch, it shows up as a "negative" for its other occurrence. We mask those logits to $-\infty$.

## 0. Setup

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # macOS: torch and faiss each bundle OpenMP; without this the kernel segfaults on import faiss

import json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

# SMOKE MODE: 1 config, 1 epoch, capped training examples - produces VALID artifacts
# (faiss.index, embeddings, tower weights, similar_items) so the webapp + notebook 05
# can be tested end-to-end. Numbers will be weak; rerun SMOKE = False for real results.
SMOKE = False

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/recsys_kindle/data'
except ImportError:
    DATA_DIR = './data'

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val   = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
test  = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))
with open(os.path.join(DATA_DIR, 'id_mappings.json')) as f:
    mappings = json.load(f)
N_USERS, N_ITEMS = len(mappings['user2idx']), len(mappings['item2idx'])

# Hyperparameters (tuned on val in Section 4)
DIM, HIDDEN   = 128, 256  # v2: more capacity (dim=128 also won the MF-BPR grid)
MAX_HIST      = 20
TAU           = 0.1     # temperature: sharper softmax -> harder training signal
BATCH         = 8192 if torch.cuda.is_available() else 4096  # more in-batch negatives = better training
USE_LOGQ      = True    # sampling-bias correction (bonus)
PAD           = N_ITEMS # padding index for short histories
RECENCY_GAMMA = 0.85    # v2: recency-weighted pooling; w ~ gamma^(steps back). 1.0 = plain mean (v1)
print(f'{N_USERS:,} users x {N_ITEMS:,} items')

Device: mps


127,585 users x 120,573 items


## 1. Build chronological user sequences

Training example = (history of previous liked items, next liked item). We keep chronological order **within** each user - predicting the past from the future would be leakage, same principle as the global time split.

In [3]:
pos_sorted = train[train.is_positive].sort_values(['user_idx', 'timestamp'])
user_seqs = {int(u): g.item_idx.to_numpy(dtype=np.int64)
             for u, g in pos_sorted.groupby('user_idx')}

# One example per (user, position k>=1): history = seq[:k] (last MAX_HIST), target = seq[k]
ex_users, ex_pos = [], []
for u, seq in user_seqs.items():
    if len(seq) >= 2:
        ex_users.append(np.full(len(seq) - 1, u))
        ex_pos.append(np.arange(1, len(seq)))
ex_users = np.concatenate(ex_users)
ex_pos   = np.concatenate(ex_pos)
if SMOKE and len(ex_users) > 1_000_000:
    keep = rng.choice(len(ex_users), size=1_000_000, replace=False)
    ex_users, ex_pos = ex_users[keep], ex_pos[keep]
print(f'{len(ex_users):,} training examples from {len(user_seqs):,} users')

# Item frequency for the log-Q correction: Q(i) ~ P(i appears as a target in a batch)
target_counts = np.bincount(pos_sorted.item_idx.to_numpy(dtype=np.int64), minlength=N_ITEMS)
logQ = torch.tensor(np.log(np.maximum(target_counts, 1) / target_counts.sum()),
                    dtype=torch.float32, device=device)

1,912,936 training examples from 126,166 users


In [4]:
def make_batch(idx):
    """idx: positions into ex_users/ex_pos -> (hist [B, MAX_HIST], mask [B, MAX_HIST], target [B])"""
    B = len(idx)
    hist = np.full((B, MAX_HIST), PAD, dtype=np.int64)
    target = np.empty(B, dtype=np.int64)
    for r, e in enumerate(idx):
        u, k = ex_users[e], ex_pos[e]
        seq = user_seqs[u]
        h = seq[max(0, k - MAX_HIST):k]
        hist[r, :len(h)] = h
        target[r] = seq[k]
    hist = torch.as_tensor(hist, device=device)
    mask = (hist != PAD).float()
    return hist, mask, torch.as_tensor(target, device=device)

## 2. The model

In [5]:
class TwoTower(nn.Module):
    def __init__(self, n_items, dim=64, hidden=128, gamma=1.0):
        super().__init__()
        # Input embedding table (row PAD is a zero pad vector). Shared by both towers:
        # the user is represented in terms of the same base item space it retrieves from.
        self.emb = nn.Embedding(n_items + 1, dim, padding_idx=n_items)
        nn.init.normal_(self.emb.weight, std=0.01)
        with torch.no_grad():
            self.emb.weight[n_items].zero_()
        self.user_mlp = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, dim))
        self.item_mlp = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, dim))
        self.gamma = gamma  # not a parameter: fixed recency decay, saved in config.json

    def user_tower(self, hist, mask):
        # v2: recency-weighted pooling. History runs oldest -> newest; the item you read
        # yesterday says more about your next read than one from months ago, and under a
        # time-based split recent signal transfers better. w_t ~ gamma^(steps back from
        # newest), gamma=1.0 recovers the plain masked mean (v1).
        pos  = mask.cumsum(1)                              # 1..n over real items
        n    = mask.sum(1, keepdim=True)
        dist = (n - pos).clamp(min=0)                      # 0 = newest, n-1 = oldest
        w    = (self.gamma ** dist) * mask
        w    = w / w.sum(1, keepdim=True).clamp(min=1e-8)
        pooled = (self.emb(hist) * w.unsqueeze(-1)).sum(1) # [B, dim]
        return F.normalize(self.user_mlp(pooled), dim=-1)

    def item_tower(self, items):
        return F.normalize(self.item_mlp(self.emb(items)), dim=-1)

model = TwoTower(N_ITEMS, DIM, HIDDEN, gamma=RECENCY_GAMMA).to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

15.565312 M parameters


## 3. Loss + training loop

In [6]:
def in_batch_ce(model, hist, mask, target):
    u = model.user_tower(hist, mask)          # [B, d]
    v = model.item_tower(target)              # [B, d]
    logits = (u @ v.T) / TAU                  # [B, B]
    if USE_LOGQ:
        logits = logits - logQ[target].unsqueeze(0)   # sampling-bias correction
    # mask false negatives: same target item appearing elsewhere in the batch
    same = target.unsqueeze(0) == target.unsqueeze(1)
    eye = torch.eye(len(target), dtype=torch.bool, device=device)
    logits = logits.masked_fill(same & ~eye, float('-inf'))
    labels = torch.arange(len(target), device=device)
    return F.cross_entropy(logits, labels)

def train_two_tower(model, epochs=5, lr=1e-3, eval_every=1):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(ex_users)
    for epoch in range(1, epochs + 1):
        t0, total = time.time(), 0.0
        model.train()
        perm = rng.permutation(n)
        for s in range(0, n, BATCH):
            hist, mask, target = make_batch(perm[s:s+BATCH])
            loss = in_batch_ce(model, hist, mask, target)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(target)
        msg = f'epoch {epoch} | loss {total/n:.4f} | {time.time()-t0:.0f}s'
        if epoch % eval_every == 0:
            r20 = quick_recall(model, truth_val, k=20, sample=20000)
            msg += f' | val Recall@20 {r20:.4f}'
        print(msg)
    return model

**Why Adam here but SGD for MF-BPR?** The spec pins SGD for MF-BPR's training loop; for the two-tower the optimizer is free, and Adam handles the MLP layers' varied gradient scales much better. (If asked: Adam ≈ SGD with per-parameter adaptive learning rates from running first/second moment estimates.)

## 4. Evaluation - same protocol, FAISS retrieval

User embedding at eval time = tower forward pass over the user's **last 20 training positives** (exactly what the API will do with PostgreSQL history). Items indexed in FAISS `IndexFlatIP`; embeddings are L2-normalized so inner product = cosine.

In [7]:
import faiss

def build_truth(part):
    p = part[part.is_positive & part.user_idx.notna() & part.item_idx.notna()]
    return p.groupby('user_idx')['item_idx'].agg(set).to_dict()

truth_val, truth_test = build_truth(val), build_truth(test)
seen_train = train.groupby('user_idx')['item_idx'].agg(set).to_dict()

def recall_at_k(recs, truth, k):
    s = [len(set(recs.get(u, [])[:k]) & rel) / len(rel) for u, rel in truth.items() if rel]
    return float(np.mean(s)) if s else 0.0

def ndcg_at_k(recs, truth, k):
    out = []
    for u, rel in truth.items():
        top = recs.get(u, [])[:k]
        dcg = sum(1.0 / np.log2(i + 2) for i, it in enumerate(top) if it in rel)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), k)))
        out.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(out))

def catalog_coverage(recs, n_items, k):
    seen = set()
    for r in recs.values():
        seen.update(r[:k])
    return len(seen) / n_items

@torch.no_grad()
def all_item_embeddings(model, batch=65536):
    model.eval()
    outs = []
    for s in range(0, N_ITEMS, batch):
        ids = torch.arange(s, min(s + batch, N_ITEMS), device=device)
        outs.append(model.item_tower(ids).cpu().numpy())
    return np.vstack(outs).astype('float32')

@torch.no_grad()
def embed_users(model, users):
    hist = np.full((len(users), MAX_HIST), PAD, dtype=np.int64)
    for r, u in enumerate(users):
        seq = user_seqs.get(u, np.array([], dtype=np.int64))[-MAX_HIST:]
        hist[r, :len(seq)] = seq
    hist_t = torch.as_tensor(hist, device=device)
    return model.user_tower(hist_t, (hist_t != PAD).float()).cpu().numpy().astype('float32')

def retrieve(model, index, users, k_retrieve=100, batch=8192):
    recs = {}
    users = [u for u in users if u in user_seqs]   # need history for the user tower
    for s in range(0, len(users), batch):
        chunk = users[s:s+batch]
        buf = max((len(seen_train.get(u, ())) for u in chunk), default=0) + 10
        _, I = index.search(embed_users(model, chunk), k_retrieve + buf)
        for u, row in zip(chunk, I):
            seen = seen_train.get(u, set())
            recs[u] = [int(i) for i in row if i not in seen][:k_retrieve]
    return recs

def quick_recall(model, truth, k=20, sample=None):
    users = list(truth.keys())
    if sample and len(users) > sample:
        users = list(rng.choice(users, size=sample, replace=False))
    index = faiss.IndexFlatIP(DIM)
    index.add(all_item_embeddings(model))
    recs = retrieve(model, index, users, k_retrieve=k)
    return recall_at_k(recs, {u: truth[u] for u in users}, k)

## 5. Train + tune

Small grid on validation (τ and lr are the sensitive knobs; batch size helps monotonically until GPU memory runs out).

In [8]:
faiss.omp_set_num_threads(1)
CONFIGS = [(0.1, 1e-3)] if SMOKE else [(0.1, 1e-3), (0.05, 1e-3)] # (0.1, 1e-3), (0.05, 1e-3), (0.1, 3e-4)
EPOCHS = 1 if SMOKE else 8   # in-batch CE keeps improving for several epochs; 8 is a good budget on a desktop GPU

GRID = []
for tau, lr in CONFIGS:
    print(f'\n=== tau={tau} lr={lr} ===')
    TAU = tau
    m = TwoTower(N_ITEMS, DIM, HIDDEN, gamma=RECENCY_GAMMA).to(device)
    m = train_two_tower(m, epochs=EPOCHS, lr=lr)
    r20 = quick_recall(m, truth_val, k=20, sample=20000)
    GRID.append({'tau': tau, 'lr': lr, 'val_Recall@20': r20, 'model': m})
    print(f'-> val Recall@20 {r20:.4f}')

pd.DataFrame([{k: v for k, v in g.items() if k != 'model'} for g in GRID]).sort_values('val_Recall@20', ascending=False)


=== tau=0.1 lr=0.001 ===


epoch 1 | loss 7.2037 | 18s | val Recall@20 0.0478


epoch 2 | loss 5.2933 | 15s | val Recall@20 0.0734


epoch 3 | loss 4.7040 | 15s | val Recall@20 0.0823


epoch 4 | loss 4.4267 | 15s | val Recall@20 0.0894


...

epoch 6 | loss 3.3619 | 15s | val Recall@20 0.0878


epoch 7 | loss 3.1046 | 15s | val Recall@20 0.0866


epoch 8 | loss 2.8921 | 15s | val Recall@20 0.0814


-> val Recall@20 0.0836


,tau,lr,val_Recall@20
0,0.10,0.001,0.091389
1,0.05,0.001,0.083650


In [9]:
best = max(GRID, key=lambda g: g['val_Recall@20'])
model, TAU = best['model'], best['tau']
print(f"Best: tau={best['tau']} lr={best['lr']} (val Recall@20={best['val_Recall@20']:.4f})")

Best: tau=0.1 lr=0.001 (val Recall@20=0.0914)


## 6. Final test evaluation & three-way comparison

In [10]:
item_emb = all_item_embeddings(model)
index = faiss.IndexFlatIP(DIM)
index.add(item_emb)

recs_test = retrieve(model, index, truth_test.keys(), k_retrieve=100)
row_tt = {'Model': 'Two-tower',
          'Recall@20': recall_at_k(recs_test, truth_test, 20),
          'Recall@50': recall_at_k(recs_test, truth_test, 50),
          'NDCG@10':   ndcg_at_k(recs_test, truth_test, 10),
          'Coverage@10': catalog_coverage(recs_test, N_ITEMS, 10)}

with open(os.path.join(DATA_DIR, 'baseline_results.json')) as f:
    rows = json.load(f)
rows = [r for r in rows if r['Model'] != 'Two-tower'] + [row_tt]
with open(os.path.join(DATA_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(rows, f, indent=2)
pd.DataFrame(rows).set_index('Model').round(4)

,Recall@20,Recall@50,NDCG@10,Coverage@10
Model,,,,
Random,0.0001,0.0004,0.0000,0.9415
Popularity,0.0141,0.0242,0.0053,0.0002
MF-BPR,0.0246,0.0461,0.0092,0.2211
Two-tower + LightGBM,0.0661,0.1070,0.0273,0.3043
Two-tower,0.0572,0.0967,0.0215,0.2996


**Interpretation.** The two-tower beats MF-BPR on test Recall@20 (0.057 vs 0.025) and popularity by ~4x, covering 30% of the catalog. The grid picked tau=0.1 (tau=0.05 overfit: val peaked then slid). The gap I keep noting is val Recall@20 0.091 vs test 0.057, the same model on a later window, which is the temporal drift the whole project runs into.

## 7. Export artifacts - this is what the webapp serves

- `item_embeddings.npy` + `faiss.index` → API startup
- `user_tower` weights + config → real-time user embedding from PostgreSQL history
- `similar_items.parquet` → precomputed top-10 cosine neighbors per item ("Similar items" + "Because you liked X" - static, so precomputed, per spec)
- candidates for train users → notebook 05 ranker training

In [11]:
ART = os.path.join(DATA_DIR, 'artifacts_two_tower')
os.makedirs(ART, exist_ok=True)

np.save(os.path.join(ART, 'item_embeddings.npy'), item_emb)
faiss.write_index(index, os.path.join(ART, 'faiss.index'))
torch.save(model.state_dict(), os.path.join(ART, 'two_tower.pt'))
with open(os.path.join(ART, 'config.json'), 'w') as f:
    json.dump({'dim': DIM, 'hidden': HIDDEN, 'max_hist': MAX_HIST, 'tau': TAU,
               'n_items': N_ITEMS, 'pad': PAD, 'gamma': RECENCY_GAMMA}, f)

# Precompute top-10 similar items (k+1 because the nearest neighbor of an item is itself)
D, I = index.search(item_emb, 11)
sim = pd.DataFrame({
    'item_idx':     np.repeat(np.arange(N_ITEMS), 10),
    'neighbor_idx': I[:, 1:].ravel(),
    'score':        D[:, 1:].ravel().astype('float32'),
})
sim.to_parquet(os.path.join(ART, 'similar_items.parquet'), index=False)
print(f'similar_items: {len(sim):,} rows')

similar_items: 1,205,730 rows
